# Omni Medical Suite — Hitti Dictionary Complete Pipeline

Run all cells to:
1. Install dependencies
2. Download pages from Archive.org
3. Run OCR (English + Arabic)
4. Extract medical glossary entries
5. Export to JSON, CSV, TXT

**Author**: DrAbdulmalek / Z.ai

In [ ]:
# %%capture
# Cell 1: Install dependencies
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-ara tesseract-ocr-eng poppler-utils > /dev/null 2>&1
!pip install -q pytesseract opencv-python-headless Pillow requests img2pdf gradio

In [ ]:
# Cell 2: Configuration
BOOK_ID = "hittisnewmedical0000hitt"
BOOK_URL = f"https://archive.org/details/{BOOK_ID}"
START_PAGE = 1
END_PAGE = 50  # Adjust as needed
OUTPUT_DIR = "./hitti_output"

import os
os.makedirs(f"{OUTPUT_DIR}/pages", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/ocr", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/glossary", exist_ok=True)
print(f"Book: {BOOK_ID}")
print(f"Pages: {START_PAGE} to {END_PAGE}")

In [ ]:
# Cell 3: Download pages via IIIF API
import requests
from pathlib import Path
import time

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

downloaded = 0
for page_num in range(START_PAGE, END_PAGE + 1):
    url = f"https://archive.org/iiif/{BOOK_ID}/page/{page_num}/full/pct:100/0/default.jpg"
    img_path = Path(f"{OUTPUT_DIR}/pages/page_{page_num:04d}.jpg")
    
    if img_path.exists():
        print(f"  [skip] Page {page_num} already exists")
        downloaded += 1
        continue
    
    try:
        resp = session.get(url, timeout=30)
        if resp.status_code == 200 and len(resp.content) > 1000:
            img_path.write_bytes(resp.content)
            downloaded += 1
            if page_num % 10 == 0:
                print(f"  Downloaded {downloaded} pages...")
    except Exception as e:
        print(f"  [error] Page {page_num}: {e}")
    
    time.sleep(0.5)  # Be respectful to Archive.org

print(f"\nTotal downloaded: {downloaded} pages")

In [ ]:
# Cell 4: Run OCR on all pages
import cv2
import pytesseract

all_texts = {}
for page_num in range(START_PAGE, END_PAGE + 1):
    img_path = Path(f"{OUTPUT_DIR}/pages/page_{page_num:04d}.jpg")
    ocr_path = Path(f"{OUTPUT_DIR}/ocr/page_{page_num:04d}.txt")
    
    if not img_path.exists():
        continue
    
    if ocr_path.exists():
        text = ocr_path.read_text(encoding="utf-8")
        all_texts[page_num] = text
        continue
    
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    binary = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    text = pytesseract.image_to_string(binary, lang="eng+ara", config="--psm 6 --oem 3")
    
    ocr_path.write_text(text, encoding="utf-8")
    all_texts[page_num] = text
    
    if page_num % 10 == 0:
        print(f"  OCR: {page_num - START_PAGE + 1}/{END_PAGE - START_PAGE + 1} pages")

print(f"\nOCR complete: {len(all_texts)} pages")

In [ ]:
# Cell 5: Extract glossary entries (7 patterns)
import re
from dataclasses import dataclass, asdict
from typing import List

@dataclass
class GlossaryEntry:
    entry_type: str
    term1: str
    term2: str
    page_num: int
    context: str = ""
    confidence: float = 0.0

def extract_entries(text, page_num):
    entries = []
    # Pattern 1: En -> Ar (comma/semicolon)
    p1 = re.compile(r'([A-Za-z][A-Za-z\s\-/]{2,50})[,;:]\s*([\u0600-\u06FF\s]{2,100})', re.MULTILINE)
    for m in p1.finditer(text):
        en, ar = m.group(1).strip(), m.group(2).strip()
        if len(en) > 2 and len(ar) > 2:
            entries.append(GlossaryEntry("en_ar", en, ar, page_num, text[max(0,m.start()-50):m.end()+50], 0.7))
    # Pattern 2: Ar -> En
    p2 = re.compile(r'([\u0600-\u06FF][\u0600-\u06FF\s]{2,50})[,;:]\s*([A-Za-z][A-Za-z\s\-/]{2,50})', re.MULTILINE)
    for m in p2.finditer(text):
        ar, en = m.group(1).strip(), m.group(2).strip()
        if len(ar) > 2 and len(en) > 2:
            entries.append(GlossaryEntry("ar_en", ar, en, page_num, text[max(0,m.start()-50):m.end()+50], 0.7))
    # Pattern 3: Bold dict format
    p3 = re.compile(r'^([A-Z][A-Z\s\-/]{1,40})\s*[\u2014\-]\s*(.+)$', re.MULTILINE)
    for m in p3.finditer(text):
        en = m.group(1).strip()
        ar_m = re.search(r'([\u0600-\u06FF]{2,100})', m.group(2))
        if ar_m:
            entries.append(GlossaryEntry("en_ar", en, ar_m.group(1).strip(), page_num, m.group(2), 0.6))
    # Pattern 4: Colon separator
    p4 = re.compile(r'([A-Za-z][A-Za-z\s\-/]{2,50})\s*:\s*([\u0600-\u06FF]{2,100})', re.MULTILINE)
    for m in p4.finditer(text):
        en, ar = m.group(1).strip(), m.group(2).strip()
        if len(en) > 2 and len(ar) > 2:
            entries.append(GlossaryEntry("en_ar", en, ar, page_num, text[max(0,m.start()-30):m.end()+30], 0.65))
    return entries

all_entries = []
for page_num, text in all_texts.items():
    entries = extract_entries(text, page_num)
    all_entries.extend(entries)

en_ar = [e for e in all_entries if e.entry_type == "en_ar"]
ar_en = [e for e in all_entries if e.entry_type == "ar_en"]
print(f"Extracted: {len(en_ar)} EN->AR + {len(ar_en)} AR->EN = {len(all_entries)} total")

In [ ]:
# Cell 6: Save to SQLite database
import sqlite3

db_path = f"{OUTPUT_DIR}/hitti_glossary.db"
conn = sqlite3.connect(db_path)
c = conn.cursor()
c.execute("CREATE TABLE IF NOT EXISTS en_ar_glossary (id INTEGER PRIMARY KEY, english_term TEXT, arabic_term TEXT, page_num INTEGER, context TEXT, confidence REAL, verified INTEGER DEFAULT 0)")
c.execute("CREATE TABLE IF NOT EXISTS ar_en_glossary (id INTEGER PRIMARY KEY, arabic_term TEXT, english_term TEXT, page_num INTEGER, context TEXT, confidence REAL, verified INTEGER DEFAULT 0)")

for e in en_ar:
    c.execute("INSERT OR IGNORE INTO en_ar_glossary (english_term, arabic_term, page_num, context, confidence) VALUES (?,?,?,?,?)",
              (e.term1, e.term2, e.page_num, e.context[:500], e.confidence))
for e in ar_en:
    c.execute("INSERT OR IGNORE INTO ar_en_glossary (arabic_term, english_term, page_num, context, confidence) VALUES (?,?,?,?,?)",
              (e.term1, e.term2, e.page_num, e.context[:500], e.confidence))
conn.commit()
conn.close()
print(f"Database saved: {db_path}")

In [ ]:
# Cell 7: Export to JSON
import json

export = {
    "en_ar": [{"english": e.term1, "arabic": e.term2, "page": e.page_num, "confidence": e.confidence} for e in en_ar],
    "ar_en": [{"arabic": e.term1, "english": e.term2, "page": e.page_num, "confidence": e.confidence} for e in ar_en],
    "metadata": {"book": "Hitti New Medical Dictionary", "pages": f"{START_PAGE}-{END_PAGE}", "total_en_ar": len(en_ar), "total_ar_en": len(ar_en)}
}

json_path = f"{OUTPUT_DIR}/glossary/hitti_glossary.json"
Path(json_path).parent.mkdir(exist_ok=True)
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"JSON exported: {json_path} ({len(en_ar)} EN->AR + {len(ar_en)} AR->EN)")

In [ ]:
# Cell 8: Export to CSV
import csv

csv_en_ar = f"{OUTPUT_DIR}/glossary/hitti_en_ar.csv"
csv_ar_en = f"{OUTPUT_DIR}/glossary/hitti_ar_en.csv"

with open(csv_en_ar, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["English", "Arabic", "Page", "Confidence"])
    for e in en_ar:
        w.writerow([e.term1, e.term2, e.page_num, e.confidence])

with open(csv_ar_en, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Arabic", "English", "Page", "Confidence"])
    for e in ar_en:
        w.writerow([e.term1, e.term2, e.page_num, e.confidence])

print(f"CSV exported: {csv_en_ar}")
print(f"CSV exported: {csv_ar_en}")

In [ ]:
# Cell 9: Sample results preview
print("=" * 60)
print("SAMPLE: English -> Arabic (first 10)")
print("=" * 60)
for e in en_ar[:10]:
    print(f"  {e.term1:<30} -> {e.term2:<30} (p.{e.page_num})")

print()
print("=" * 60)
print("SAMPLE: Arabic -> English (first 10)")
print("=" * 60)
for e in ar_en[:10]:
    print(f"  {e.term1:<30} -> {e.term2:<30} (p.{e.page_num})")

In [ ]:
# Cell 10: Download results as ZIP
import zipfile
from google.colab import files

zip_path = f"{OUTPUT_DIR}/hitti_glossary_export.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path(f"{OUTPUT_DIR}/glossary").glob("*"):
        zf.write(f, f.name)
    zf.write(f"{OUTPUT_DIR}/hitti_glossary.db", "hitti_glossary.db")

print(f"ZIP created: {zip_path}")
files.download(zip_path)